<a href="https://colab.research.google.com/github/Shanta6039/finance-assistant-/blob/main/finance_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

   # Student Budget Checker
   ISYS2001 Assessment 2 — Programming Project
   Author: Shanta

## Step 1: Understand the problem

**1. Who is my user?**
My target users are university students aged 18–25 who have limited income from part-time jobs, family support or scholarships. They often have to manage expenses such as rent, groceries, transport, tuition fees and entertainment.

**2. What is their problem?**
Students often struggle to manage their money because they do not track their daily spending or follow a budget. As a result, they may run out of money before the end of the month without knowing where it went.

**3. Why does it matter?**
Poor money management can cause financial stress, unpaid bills and difficulty covering essential expenses. It can also prevent students from saving money and force them to borrow from friends or family.

**4. How will my app help?**
My Student Budget Checker lets students upload a CSV file of their transactions. It adds up their spending in each category, compares it with their monthly budget, and warns them when they spend more than their budget. Students can also chat with an AI assistant that gives personalised saving tips based on their real spending numbers.

## Step 2: Inputs and outputs

**Inputs (what goes into my app)**
- **Transactions CSV file:** a file of the student's spending with the columns date, description, category and amount (e.g. groceries, transport, entertainment).
- **Monthly budget per category:** how much the student plans to spend in each category each month.
- **Chat questions:** questions the student types to the AI assistant (e.g. "Where am I spending too much?").

**Outputs (what comes out of my app)**
- **Spending summary:** total spending in each category and overall.
- **Budget status:** whether each category is within or over budget.
- **Remaining balance:** how much money is left in each category (budget − spent), or how much the student went over.
- **Budget alerts:** warnings when the student has gone over budget in a category.
- **Saving tips:** personalised suggestions from the AI assistant, based on the student's real spending numbers.

**What could go wrong with the inputs**
- No file is uploaded, or the file is empty.
- The CSV is missing a column or has an amount that is not a number.
- A budget is negative, zero or not a number.
- A category in the CSV has no budget set.
- The AI service (Gemini) is busy or not responding.

## Step 3: Worked example by hand

**Sample transactions:**

| date  | description | category      | amount |
|-------|-------------|---------------|--------|
| 01/09 | Coles       | Groceries     | 45.50  |
| 03/09 | Transperth  | Transport     | 20.00  |
| 05/09 | Netflix     | Entertainment | 16.99  |
| 10/09 | Woolworths  | Groceries     | 62.30  |
| 15/09 | Movies      | Entertainment | 25.00  |
| 20/09 | Transperth  | Transport     | 20.00  |

**Budgets:** Groceries \$100, Transport \$50, Entertainment \$30

**1. Total spent per category:**
- Groceries = 45.50 + 62.30 = 107.80
- Transport = 20.00 + 20.00 = 40.00
- Entertainment = 16.99 + 25.00 = 41.99

**2. Overall total:** 107.80 + 40.00 + 41.99 = 189.79

**3. Budget status:**
- Groceries: spent 107.80 vs budget 100 → **OVER** by 107.80 − 100 = **7.80**
- Transport: spent 40.00 vs budget 50 → **OK**, 50 − 40.00 = **10.00 left**
- Entertainment: spent 41.99 vs budget 30 → **OVER** by 41.99 − 30 = **11.99**

**Summary:** The student spent \$189.79 in total. They went over budget in Groceries (by \$7.80) and Entertainment (by \$11.99), but stayed within budget in Transport with \$10.00 left.

## Step 4: Pseudocode

1. Start the Student Budget Checker app.
2. Ask the user to upload their transactions CSV file.
3. If the file is missing or empty, show an error message and stop.
4. Check whether the file contains the required columns and valid expense amounts. If any information is missing or invalid, show an error message.
5. Ask the user to enter a monthly budget for each expense category.
6. If a budget is negative, zero or invalid, show an error message and ask the user to enter it again.
7. Set the total spending for each category to 0.
8. For each transaction, add the expense amount to the relevant category's total.
9. Calculate the total expenses (sum of all category totals) and the remaining balance (total budget − total expenses).
10. For each category:
    * If no budget is set, display "No budget set".
    * If spending exceeds the budget, display "OVER" and calculate the excess amount.
    * Otherwise, display "WITHIN BUDGET" and calculate the remaining amount.
11. Display the spending summary, remaining balance and budget status to the user.
12. Send the spending summary to the AI assistant.
13. The AI assistant provides saving tips based on the user's spending patterns.
14. When the user asks a financial question, the AI assistant answers based on their actual spending and budget.
15. If the AI assistant is unavailable, retry the request and display a friendly error message if it still fails.
16. End.

In [ ]:
# Install the Gemini library (only needed once per session)
!pip install -q google-genai

import time                           # lets us wait between tries
from google.colab import userdata     # reads Colab Secrets
from google import genai              # Google's Gemini library
from google.genai import errors       # Gemini's error types

# Get the key from Secrets and connect to Gemini
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# Models to try, in order. If one is busy, we try the next.
models_to_try = ["gemini-flash-latest", "gemini-flash-lite-latest"]

def ask_gemini(question):
    """Send a question to Gemini. Retry if busy, and never crash."""
    for model in models_to_try:
        for attempt in range(1, 4):          # up to 3 tries per model
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=question
                )
                print(f"✅ Success using {model} (try {attempt})")
                return response.text
            except errors.ServerError:
                # 503: Google is busy. Wait, then try again.
                print(f"⏳ {model} is busy (try {attempt}), waiting 5 seconds...")
                time.sleep(5)
            except errors.ClientError as e:
                # 4xx: a problem on our side, e.g. wrong key or unknown model
                print(f"❌ Problem with {model}: {e}")
                break                         # skip to the next model
    return "Sorry, the assistant is busy right now. Please try again in a few minutes."

# Test it
print(ask_gemini("Say hello to Shanta in one short sentence."))

⏳ gemini-flash-latest is busy (try 1), waiting 5 seconds...
⏳ gemini-flash-latest is busy (try 2), waiting 5 seconds...
⏳ gemini-flash-latest is busy (try 3), waiting 5 seconds...
✅ Success using gemini-flash-lite-latest (try 1)
Hello, Shanta, I hope you're having a wonderful day!
